## Local Inference on GPU 
Model page: https://huggingface.co/microsoft/VibeVoice-ASR

⚠️ If the generated code snippets do not work, please open an issue on either the [model repo](https://huggingface.co/microsoft/VibeVoice-ASR)
			and/or on [huggingface.js](https://github.com/huggingface/huggingface.js/blob/main/packages/tasks/src/model-libraries-snippets.ts) 🙏

In [ ]:
# Load model weights directly
import json
from pathlib import Path

import torch
from huggingface_hub import hf_hub_download
from safetensors import safe_open

model_path = "microsoft/VibeVoice-ASR"
index_path = Path(hf_hub_download(model_path, "model.safetensors.index.json"))
weight_map = json.loads(index_path.read_text())["weight_map"]
shard_names = list(dict.fromkeys(weight_map.values()))
selected = None
selected_tensor = None

for shard_name in shard_names:
    shard_path = hf_hub_download(model_path, shard_name)
    with safe_open(shard_path, framework="pt", device="cuda:0") as weights:
        keys = list(weights.keys())
        preferred = [k for k in keys if k.endswith(".bias") or "norm" in k.lower()]
        selected = preferred[0] if preferred else keys[0]
        selected_tensor = weights.get_tensor(selected)
        break

assert selected_tensor is not None
with torch.no_grad():
    value = selected_tensor.float().mean().item()
print("loaded tensor =", selected)
print("shape =", tuple(selected_tensor.shape), "device =", selected_tensor.device, "mean =", round(value, 6))
